In [ ]:
"""
Load a frequency-domain susceptibility spectrum and fit it to
a frequency-domain KWW model using HN-style weighted-linear fitting
with adaptive oscillatory quadrature for the KWW Fourier transform.

Input CSV should contain:
omega_rad_per_s
Chi_avg

Outputs:
*_KWWfreqfit_params.csv KWW fit parameters and uncertainties
*_KWWfreqfit_curve.csv Numerical fitted KWW curve
*_KWWfreqfit_semilogx.png KWW fit plot
"""

# ============================================================
# Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.integrate import quad


# ============================================================
# User Inputs
# ============================================================

# Input susceptibility spectrum
SPECTRA_FILE = r"C:\path\to\ChiLossData_omega.csv"

# Column names in the input CSV
OMEGA_COL = "omega_rad_per_s"
CHI_COL = "Chi_avg"

# Output files are saved next to the input file
SAVE_PREFIX = str(
    Path(SPECTRA_FILE).with_suffix("")
)


# ============================================================
# Numerical Integration Settings
# ============================================================

# Absolute and relative tolerances for the KWW Fourier transform
KWW_EPSABS = 1e-10
KWW_EPSREL = 1e-8

# Maximum number of adaptive subdivisions
KWW_LIMIT = 300

# Maximum number of Fourier cycles for the infinite-interval
# sine-weighted quadrature
KWW_LIMLST = 300

# Number of Chebyshev moments used by weighted quadrature
KWW_MAXP1 = 200

# Split the integral such that the oscillatory phase in the
# first interval remains small
KWW_SPLIT_PHASE = 0.25


# ============================================================
# KWW Fourier Transform
# ============================================================

def kww_loss_transform_scalar(x, beta):
    """
    Evaluate the dimensionless KWW loss transform

        I(x, beta)
            = integral_0^infinity [
                  beta * u^(beta-1) * exp(-u^beta)
                  * sin(x*u)
              ] du

    where

        x = omega * tau.

    For beta < 1, the KWW kernel is integrably singular at u = 0.
    The endpoint region is therefore transformed using

        y = u^beta,

    for which

        beta * u^(beta-1) du = dy.

    The remaining oscillatory tail is evaluated using
    sine-weighted adaptive quadrature.
    """

    x = float(x)
    beta = float(beta)

    if not np.isfinite(x) or not np.isfinite(beta):
        return np.nan

    if x < 0:
        raise ValueError(
            "Dimensionless frequency x must be nonnegative."
        )

    if not (0 < beta <= 1):
        raise ValueError(
            "KWW exponent beta must satisfy 0 < beta <= 1."
        )

    if x == 0:
        return 0.0

    # --------------------------------------------------------
    # Select splitting point
    # --------------------------------------------------------

    # Keep x * u_split <= KWW_SPLIT_PHASE.
    # The cap u_split <= 1 avoids unnecessarily enlarging the
    # endpoint interval at low frequency.

    u_split = min(
        1.0,
        KWW_SPLIT_PHASE / x
    )

    y_split = u_split ** beta

    # --------------------------------------------------------
    # Part 1: endpoint region
    #
    # u = 0 ... u_split
    #
    # Transform:
    #
    # y = u^beta
    #
    # giving
    #
    # integral exp(-y) sin[x y^(1/beta)] dy
    # --------------------------------------------------------

    def endpoint_integrand(y):

        if y == 0:
            return 0.0

        u_value = y ** (1.0 / beta)

        return (
            np.exp(-y)
            * np.sin(x * u_value)
        )

    endpoint_value, _ = quad(
        endpoint_integrand,
        0.0,
        y_split,
        epsabs=KWW_EPSABS,
        epsrel=KWW_EPSREL,
        limit=KWW_LIMIT
    )

    # --------------------------------------------------------
    # Part 2: oscillatory tail
    #
    # u_split ... infinity
    #
    # scipy.integrate.quad with weight="sin" evaluates
    #
    # integral f(u) sin(x*u) du
    # --------------------------------------------------------

    def kww_kernel(u):

        return (
            beta
            * u ** (beta - 1.0)
            * np.exp(-(u ** beta))
        )

    tail_value, _ = quad(
        kww_kernel,
        u_split,
        np.inf,
        weight="sin",
        wvar=x,
        epsabs=KWW_EPSABS,
        epsrel=KWW_EPSREL,
        limit=KWW_LIMIT,
        limlst=KWW_LIMLST,
        maxp1=KWW_MAXP1
    )

    transform_value = endpoint_value + tail_value

    if not np.isfinite(transform_value):
        raise RuntimeError(
            "Nonfinite result encountered while evaluating "
            f"KWW transform at x={x:.6g}, beta={beta:.6g}."
        )

    return transform_value


def kww_loss_transform(x, beta):
    """
    Vectorized interface to the scalar KWW loss transform.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    original_shape = x.shape
    x_flat = x.ravel()

    result_flat = np.fromiter(
        (
            kww_loss_transform_scalar(xi, beta)
            for xi in x_flat
        ),
        dtype=float,
        count=x_flat.size
    )

    return result_flat.reshape(
        original_shape
    )


def kww_chi_loss(
    omega,
    A,
    tau_fit,
    beta,
    b
):
    """
    Frequency-domain KWW loss susceptibility.

        chi''(omega)
            = A * I(omega * tau_fit, beta) + b
    """

    omega = np.asarray(
        omega,
        dtype=float
    )

    dimensionless_frequency = (
        omega * tau_fit
    )

    transform = kww_loss_transform(
        dimensionless_frequency,
        beta
    )

    return (
        A * transform
        + b
    )


# ============================================================
# Helper Functions
# ============================================================

def r2_lin(y, yhat):
    """
    Linear-space coefficient of determination.
    """

    y = np.asarray(
        y,
        dtype=float
    )

    yhat = np.asarray(
        yhat,
        dtype=float
    )

    ss_res = np.sum(
        (y - yhat) ** 2
    )

    ss_tot = np.sum(
        (y - np.mean(y)) ** 2
    )

    if ss_tot <= 0:
        return np.nan

    return (
        1.0
        - ss_res / ss_tot
    )


def load_spectrum(
    spectra_file,
    omega_col,
    chi_col
):
    """
    Load and clean the susceptibility spectrum.
    """

    df = pd.read_csv(
        spectra_file
    )

    if omega_col not in df.columns:
        raise KeyError(
            f"Column '{omega_col}' was not found in:\n"
            f"{spectra_file}"
        )

    if chi_col not in df.columns:
        raise KeyError(
            f"Column '{chi_col}' was not found in:\n"
            f"{spectra_file}"
        )

    omega = df[
        omega_col
    ].to_numpy(dtype=float)

    chi = df[
        chi_col
    ].to_numpy(dtype=float)

    mask = (
        np.isfinite(omega)
        & np.isfinite(chi)
        & (omega > 0)
        & (chi > 0)
    )

    omega = omega[mask]
    chi = chi[mask]

    if omega.size < 4:
        raise ValueError(
            "Fewer than four valid positive spectrum "
            "points were found."
        )

    # Sort frequencies for consistent plotting/output
    order = np.argsort(
        omega
    )

    return (
        omega[order],
        chi[order]
    )


# ============================================================
# Frequency-Domain Fitting
# ============================================================

def fit_kww_frequency(
    omega,
    chi
):
    """
    Fit the frequency-domain KWW model.

    Using sigma = chi causes curve_fit to minimize approximately

        [(chi_data - chi_fit) / chi_data]^2,

    giving approximately equal weighting to relative residuals
    across the spectrum.
    """

    peak_index = np.argmax(
        chi
    )

    omega_peak = omega[
        peak_index
    ]

    # Initial guess
    p0 = [
        np.max(chi),                  # A
        1.0 / omega_peak,             # tau_fit
        0.5,                          # beta
        max(0.0, 0.1 * np.min(chi))  # baseline
    ]

    # Parameter bounds
    bounds = (
        [
            1e-12,   # A
            1e-12,   # tau_fit
            0.05,    # beta
            0.0      # baseline
        ],
        [
            np.inf,  # A
            1e8,     # tau_fit
            1.0,     # beta
            np.inf   # baseline
        ]
    )

    # Relative-style fitting weights
    sigma = chi.copy()

    positive_chi = chi[
        chi > 0
    ]

    if positive_chi.size == 0:
        raise ValueError(
            "The susceptibility spectrum contains "
            "no positive values."
        )

    sigma[
        sigma <= 0
    ] = np.median(
        positive_chi
    )

    popt, pcov = curve_fit(
        kww_chi_loss,
        omega,
        chi,
        p0=p0,
        bounds=bounds,
        sigma=sigma,
        absolute_sigma=False,
        maxfev=200000
    )

    chi_fit = kww_chi_loss(
        omega,
        *popt
    )

    r2 = r2_lin(
        chi,
        chi_fit
    )

    return (
        popt,
        pcov,
        chi_fit,
        r2
    )


# ============================================================
# Save Results
# ============================================================

def save_fit_params(
    save_prefix,
    popt,
    pcov,
    r2
):
    """
    Save fitted parameters and approximate
    one-standard-deviation uncertainties.
    """

    A, tau_fit, beta, b = popt

    if (
        pcov is not None
        and pcov.shape == (4, 4)
        and np.all(
            np.isfinite(
                np.diag(pcov)
            )
        )
    ):
        parameter_errors = np.sqrt(
            np.maximum(
                np.diag(pcov),
                0.0
            )
        )

    else:
        parameter_errors = np.full(
            4,
            np.nan
        )

    (
        A_err,
        tau_fit_err,
        beta_err,
        b_err
    ) = parameter_errors

    params_df = pd.DataFrame(
        {
            "A": [A],
            "A_std": [A_err],

            "tau_fit_s": [tau_fit],
            "tau_fit_std_s": [tau_fit_err],

            "beta": [beta],
            "beta_std": [beta_err],

            "baseline_b": [b],
            "baseline_b_std": [b_err],

            "R2_linear": [r2]
        }
    )

    out_params = (
        save_prefix
        + "_KWWfreqfit_params.csv"
    )

    params_df.to_csv(
        out_params,
        index=False
    )

    print(
        "Saved fit parameters to:\n"
        f"{out_params}"
    )


def save_fit_curve(
    save_prefix,
    omega_smooth,
    chi_smooth
):
    """
    Save smooth fitted KWW spectrum.
    """

    curve_df = pd.DataFrame(
        {
            "omega_rad_per_s":
                omega_smooth,

            "Chi_KWW_fit":
                chi_smooth
        }
    )

    out_curve = (
        save_prefix
        + "_KWWfreqfit_curve.csv"
    )

    curve_df.to_csv(
        out_curve,
        index=False
    )

    print(
        "Saved fitted curve to:\n"
        f"{out_curve}"
    )


# ============================================================
# Plot Results
# ============================================================

def plot_fit(
    save_prefix,
    omega,
    chi,
    popt,
    r2
):
    """
    Plot and save the fitted susceptibility spectrum.
    """

    A, tau_fit, beta, b = popt

    omega_smooth = np.logspace(
        np.log10(
            omega.min()
        ),
        np.log10(
            omega.max()
        ),
        512
    )

    chi_smooth = kww_chi_loss(
        omega_smooth,
        *popt
    )

    save_fit_curve(
        save_prefix,
        omega_smooth,
        chi_smooth
    )

    plt.figure(
        figsize=(5, 5)
    )

    plt.semilogx(
        omega,
        chi,
        "o",
        ms=4,
        label="Spectrum"
    )

    plt.semilogx(
        omega_smooth,
        chi_smooth,
        "-",
        lw=2,
        label=(
            rf"KWW fit: "
            rf"$\beta={beta:.3f}$, "
            rf"$\tau={tau_fit:.3g}$ s, "
            rf"$R^2={r2:.3f}$"
        )
    )

    plt.xlabel(
        r"Angular frequency $\omega$ (rad/s)"
    )

    plt.ylabel(
        r"$\chi''(\omega)$ (a.u.)"
    )

    plt.legend()
    plt.tight_layout()

    fig_path = (
        save_prefix
        + "_KWWfreqfit_semilogx.png"
    )

    plt.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        "Saved fit plot to:\n"
        f"{fig_path}"
    )


# ============================================================
# Main
# ============================================================

def main():

    omega, chi = load_spectrum(
        SPECTRA_FILE,
        OMEGA_COL,
        CHI_COL
    )

    print(
        f"Loaded {omega.size} valid spectrum points."
    )

    print(
        "Evaluating KWW transform using "
        "adaptive oscillatory quadrature..."
    )

    popt, pcov, chi_fit, r2 = (
        fit_kww_frequency(
            omega,
            chi
        )
    )

    A, tau_fit, beta, b = popt

    if (
        pcov is not None
        and pcov.shape == (4, 4)
        and np.all(
            np.isfinite(
                np.diag(pcov)
            )
        )
    ):
        errors = np.sqrt(
            np.maximum(
                np.diag(pcov),
                0.0
            )
        )

    else:
        errors = np.full(
            4,
            np.nan
        )

    (
        A_err,
        tau_err,
        beta_err,
        b_err
    ) = errors

    print(
        "\n===== Frequency-domain KWW fit ====="
    )

    print(
        f"A        = {A:.6g} "
        f"+/- {A_err:.3g}"
    )

    print(
        f"tau_fit  = {tau_fit:.6g} "
        f"+/- {tau_err:.3g} s"
    )

    print(
        f"beta     = {beta:.6g} "
        f"+/- {beta_err:.3g}"
    )

    print(
        f"b        = {b:.6g} "
        f"+/- {b_err:.3g}"
    )

    print(
        f"R2       = {r2:.6g}"
    )

    save_fit_params(
        SAVE_PREFIX,
        popt,
        pcov,
        r2
    )

    plot_fit(
        SAVE_PREFIX,
        omega,
        chi,
        popt,
        r2
    )

    print(
        "\nAnalysis complete."
    )


if __name__ == "__main__":
    main()